In [2]:
import os
import gradio as gr
import requests
import inspect
import pandas as pd
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import Tool,tool
import traceback

ModuleNotFoundError: No module named 'gradio'

In [1]:
# --- Constants ---
DEFAULT_API_URL = "https://agents-course-unit4-scoring.hf.space"


if "GEMINI_API_KEY" in os.environ:
    api_key = os.environ["GEMINI_API_KEY"]
else:
    api_key = input("Enter your GEMINI API key: ")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", #gemini-3.5-flash-lite
    google_api_key=api_key,
    thinking_level="high")

NameError: name 'os' is not defined

# Tools

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages

# Generate the AgentState 
#class AgentState(TypedDict):
#    messages: Annotated[list[AnyMessage], add_messages]

from typing import Annotated, Optional #TypedDict,
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage


class AgentState(TypedDict):
    # The input document
    input_file: Optional[str]  # Contains file path, type (PNG)
    messages: Annotated[list[AnyMessage], add_messages]


def fntest(query: str):
    """A tool for testing

    Returns:
        string: test string
    """
    # Placeholder for tool testing logic
    return "Tool test executed."

tooltest = Tool(
    name="fntest",
    func=fntest,
    description=""
)



'''
def video_assistant_from_url(video_url: str, question: str) -> str:
    """Analyze a YouTube video to answer a question about its contents.

    Provide both the YouTube URL and the specific question to answer.

    Args:
        video_url (str): url for the video
        question (str): The context/question to be answered

    Returns:
        str: _description_
    """
    sys_prompt = SystemMessage(content=(
        "You are a video interpretation assistant. Answer based on the "
        "video's visible or audible content. If you cannot determine something, say so."
    ))
    user_message = HumanMessage(
        content=f"Video URL: {video_url}\n\nQuestion: {question}"
    )

    response = llm.invoke([sys_prompt, user_message])
    return response.content

video_assistant = Tool(
    name="video_assistant_from_url",
    func=video_assistant_from_url,
    description=""
)
'''

'\ndef video_assistant_from_url(video_url: str, question: str) -> str:\n    """Analyze a YouTube video to answer a question about its contents.\n\n    Provide both the YouTube URL and the specific question to answer.\n\n    Args:\n        video_url (str): url for the video\n        question (str): The context/question to be answered\n\n    Returns:\n        str: _description_\n    """\n    sys_prompt = SystemMessage(content=(\n        "You are a video interpretation assistant. Answer based on the "\n        "video\'s visible or audible content. If you cannot determine something, say so."\n    ))\n    user_message = HumanMessage(\n        content=f"Video URL: {video_url}\n\nQuestion: {question}"\n    )\n\n    response = llm.invoke([sys_prompt, user_message])\n    return response.content\n\nvideo_assistant = Tool(\n    name="video_assistant_from_url",\n    func=video_assistant_from_url,\n    description=""\n)\n'

### Math tools

In [4]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

@tool
def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b

### Websearch tools

In [5]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import ArxivLoader
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_community.document_loaders import WebBaseLoader
from langchain_tavily import TavilySearch
from ddgs import DDGS
from ddgs.exceptions import DDGSException


@tool
def WikiLoadTool(query: str) -> str:
    """Load a Wikipedia page based on a query and return its content.
    args:
        query (str): The search query for the Wikipedia page.
    returns:
        str: The content of the loaded Wikipedia page or a message if no content is found.
    """
    loader = WikipediaLoader(query=query)
    docs = loader.load()
    return docs[0].page_content if docs else "No content found."

@tool
def ArxivLoadTool(query: str) -> str:
    """Load an Arxiv paper based on a query and return its content.
    args:
        query (str): The search query for the Arxiv paper.
    returns:
        str: The content of the loaded Arxiv paper or a message if no content is found.
    """
    loader = ArxivLoader(query=query)
    docs = loader.load()
    return docs[0].page_content if docs else "No content found."

@tool
def websearch_tool(query: str) -> str:
    """Search the web and return the results of the query.

    args:
        query (str): The search query for the web.
    returns:
        str: The search results or a message if no results are found.
    """
    try:
        results = DDGS(timeout=10).text(query, max_results=2)
        if not results:
            return "Search returned no results. Try a different query."

        return results
    except DDGSException as e:
        return f"Search failed or returned no results: {e}. Try a different query."


@tool
def visit_website_tool(url: str) -> str:
    """Visit a website and return its content, when you have a URL.

    Read a webpage when the question provides a direct URL,
    or when a specific URL must be opened. If the page has already
    been retrieved and contains the needed information, do not
    call this tool again for that page.
    
    args:
        url (str): The URL of the website to visit."""
    loader = WebBaseLoader(url)
    docs = loader.load()
    return docs[0].page_content if docs else "No content found."
    

C:\Users\fredd\AppData\Local\Temp\ipykernel_15324\3104461507.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
@tool
def search_and_read_web_tool(query: str) -> str:
    """Search the web, then read the pages returned by the search.

    Use this as the default and usually only tool for web research
    when no direct URL is provided. The result includes page content,
    titles, and URLs. Do not call websearch_tool separately first.
    
    args:
        query (str): The query to search for.
    returns:
        str: Content of the first webpages in the search
    """
    try:
        results = websearch_tool.invoke({"query": query})
        print(results)
    except Exception as e:
        return f"Web search failed: {e}"

    if not isinstance(results, list) or not results:
        return f"No usable search results. Search returned: {results}"

    pages = []
    url = results[0].get("href")
    if url:
        try:
            content = visit_website_tool.invoke({"url": url})
            pages.append(content)
        except Exception as e:
            pages.append(f"Could not read {url}: {e}")

    return "\n\n".join(pages) if pages else "No result pages could be read."

### Read tools

In [7]:
from pathlib import Path
from urllib.parse import unquote, urlparse
import mimetypes
import requests
from langchain_core.tools import tool


@tool
def download_file_from_url_tool(url: str) -> str:
    """Download a file from a URL and return its local path.
    args:
        url (str): The URL of the file to download.
    returns:
        str: The local path of the downloaded file or a message if the download fails.
    """
    try:
        response = requests.get(url, timeout=30, stream=True)
        response.raise_for_status()

        filename = Path(unquote(urlparse(url).path)).name or "downloaded_file"
        print(filename)

        content_type = response.headers.get("Content-Type", "").split(";")[0].lower()
        print(content_type)


        if content_type == "text/plain" and "csv" in url:
            extension = ".csv"
        else:
            extension = mimetypes.guess_extension(content_type)

        print(extension)

        if extension and not mimetypes.guess_type(filename)[0]:
            filename += extension

        output_path = Path("downloads") / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

        return f"Downloaded file to: {output_path.resolve()}"
    except requests.RequestException as e:
        return f"Download failed: {e}"
    except OSError as e:
        return f"Could not save the file: {e}"

In [8]:
from langchain_community.document_loaders import PyPDFLoader


@tool
def read_pdf_from_url_tool(url: str) -> str:
    """Read a PDF from a URL and return its content.
    args:
        url (str): The URL of the PDF
    returns:
        str: The content of the PDF or a message if no content is found.
    """

    try:
        loader = PyPDFLoader(url)
        docs = loader.load()
        return docs[0].page_content if docs else "No content found."
    except Exception as e:
        return f"Error downloading or reading PDF: {e}"

@tool 
def read_text_tool(file_path: str) -> str:
    """Read a text file (python, script, text etc.) and return its content
    args:
        file_path (str): The path to the  file.
    returns:
            str: The content of the file or a message if no content is found.
    """
    try:
        path = Path(file_path)

        if not path.is_file():
            return f"File not found: {file_path}"

        # Replace invalid UTF-8 characters so the tool can still read the file.
        content = path.read_text(encoding="utf-8", errors="replace")
        return content if content.strip() else "The file is empty."

    except OSError as e:
        return f"Error reading file: {e}"

@tool
def read_csv_tool(file_path: str) -> str:
    """Read a CSV file and return its content.
    args:
        file_path (str): The path to the CSV file.
    returns:
        str: The content of the CSV file or a message if no content is found.
    """
    try:
        # Check ifor meatadataa (#)
        skiprows = 0

        with open(file_path, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.lstrip().startswith("#") or not line.strip():
                    skiprows += 1
                else:
                    break

        # Load dataframe
        df = pd.read_csv(file_path, skiprows=skiprows)
        return f"Loaded CSV file of size {len(df)}, {len(df.columns)}. Content:\n{df.to_string(index=False)}"

    except Exception as e:
        return f"Error reading CSV file: {e}"

@tool
def read_excel_tool(file_path: str) -> str:
    """Read an Excel file and return its content.
    args:
        file_path (str): The path to the excel file.
    returns:
        str: The content of the excel file or a message if no content is found.
    """
    try:
        df = pd.read_excel(file_path)
        return f"Loaded Excel file of size {len(df)}, {len(df.columns)}. Content:\n{df.to_string(index=False)}"
        
    except Exception as e:
        return f"Error reading Excel file: {e}"


### Image tool

In [9]:
import base64

@tool
def extract_text_from_image_assistant(image_path: str) -> str:
    """Extract text from the image

    Args:
        image_path (str): path to the image

    Returns:
        str: extracted text from the image
    """
    try:
        path = Path(image_path)
        mime_type = mimetypes.guess_type(path.name)[0] or "image/jpeg"
        image_data = base64.b64encode(path.read_bytes()).decode("utf-8")

        response = llm.invoke([
            HumanMessage(content=[
                {"type": "text", "text": "Extract all readable text from this image."},
                {
                    "type": "image",
                    "source_type": "base64",
                    "mime_type": mime_type,
                    "data": image_data,
                },
            ])
        ])

        return response.content
    except Exception as e:
            return f"Could not analyze the image: {e}"

### MP3 tool

In [10]:
from langchain_core.tools import tool

@tool
def analyze_mp3_tool(audio_path: str) -> str:
    """Analyze an MP3 and return a transcript or answer about its contents.
    
    Args:
        audio_path (str): path to the mp3
    
    Returns:
        str: extracted text from the mp3"""
    try:
        path = Path(audio_path)

        if not path.is_file():
            return f"Audio file not found: {audio_path}"

        audio_data = base64.b64encode(path.read_bytes()).decode("utf-8")

        response = llm.invoke([
            HumanMessage(content=[
                {
                    "type": "text",
                    "text": (
                        "Transcribe the spoken words in this audio verbatim. "
                        "Do not describe or summarize it. Return only the transcript."
                    ),
                },
                {
                    "type": "audio",
                    "base64": audio_data,
                    "mime_type": "audio/mpeg",
                },
            ])
        ])

        return response.text
    except Exception as e:
        return f"Could not analyze the MP3: {e}"

# Agent

In [11]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode

def fntest(query: str):
    """A tool for testing

    Returns:
        string: test string
    """
    # Placeholder for tool testing logic
    return "Tool test executed."

tooltest = Tool(
    name="fntest",
    func=fntest,
    description=""
)


def build_graph():
    """Build a graph

    Returns:
        _type_: _description_
    """
    
    tools = [visit_website_tool,
             search_and_read_web_tool, 
             read_pdf_from_url_tool, 
             download_file_from_url_tool, 
             read_text_tool,
             read_csv_tool, 
             read_excel_tool, 
             extract_text_from_image_assistant,
             analyze_mp3_tool,
             add, subtract, multiply, divide] # ADD TOOLS
    
    chat_with_tools = llm.bind_tools(tools)



    def assistant(state: AgentState):
        return {"messages": [chat_with_tools.invoke(state["messages"])]}

    ## The graph for 
    builder = StateGraph(AgentState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))


    # Define edges: these determine how the control flow moves
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges(
        "assistant",
        # If the latest message requires a tool, route to tools
        # Otherwise, provide a direct response
        tools_condition
    )
    builder.add_edge("tools", "assistant")

    return builder.compile()



In [13]:
ag = build_graph()

In [16]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash", #gemini-3.5-flash-lite
    google_api_key=api_key,
    temperature=0.3
)
analyze_mp3_tool.invoke("downloads/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3.mp3")

KeyboardInterrupt: 

In [ ]:

extract_text_from_image.invoke("downloads/particle_sidm_model.png")

In [ ]:
search_and_read_web_tool.invoke("mercedes sosa discography")

In [ ]:
download_file_from_url_tool.invoke("https://zenodo.org/api/records/15118128/files/Coma%20Data.csv/content")

In [ ]:
read_csv_tool.invoke("downloads/content.csv")[:500]

In [ ]:
visit_website_tool.invoke("https://zenodo.org/api/records/15118128/files/Coma%20Data.csv/content")[:500]

In [ ]:
websearch_tool.invoke("How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)?")

In [14]:
inputs = {
    "messages": [
        HumanMessage(content=(
            """How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)?
            Web browsing rules:
            - Use the minimum number of tool calls necessary. Do not browse further just because additional research is possible."""
        ))
    ]
}

for update in ag.stream(
    inputs,
    config={"recursion_limit": 12},
    stream_mode="updates",
):
    print(update)

{'assistant': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_and_read_web_tool', 'arguments': '{"query": "\\"Mercedes Sosa\\" discography studio albums"}'}, '__gemini_function_call_thought_signatures__': {'call_3315884': 'Eo0ECooEAWkUfRNhcc3vQpyebS0nVyfstvbELakTEaYzsWQPc+UjWSA7mEis2M3/WbgFpwzf8oQSJdYtqYkB0jQFr1MaL8fbnTpsanlCkPfOyovU6yns53UUTExVOwvn1DRjoWDDcBDw7SjAZzxy6HIRmQ8C5TM/L6UaEcPhNE4u7kBR1WpFihmXTC80sLrikh886LqLoaOOoiJt2Q1OxSdXiGVTxopmvQehOOThcxTftWSiACAWOsYe4I/Wp48sEQsBelRqTRpPRfI7rsUW4Dnk9LmON2A4ip6j52k38SvPfaKkihfTpKTgNiSD6her0E1ltbHczkIiKXYqLDyerBCXnyGcoGtR0uu6+HzWHxBUxWtBUtJNRPfXHq6BeqLiVlayj7cpSzRzAnvt2vJwhZYb0P+6Bd7Nqz2lNlBpHROqobo3jN3nzGl209Gua0S+Oj/fc0HktoDWgZO0q6iYbY26ybgOuvQh06h7GGAT5jO6+TmGO7zhOj4sXjfRraSeCEc/r6cZ92rKL6Ct3Ire6wkc2pjFYpoNYDoI7cIvrXxOPt8z8x/onRA0HU6UDoxmXMCEVTudJcvYNEppBrBrI6ueZCBaxhCmhqdEWQQ1fsND/DKS1yoKkgQtg6jW3TRR6Id9t1CZSEU6fAx3kcRfxSaKqVU4J2yu6qINKB4zbs2/EIc0X1BtEapywACzuuqF'}}, response_metadata={'finish_

RemoteProtocolError: Server disconnected without sending a response.

In [ ]:
import langchain
langchain.debug = True

ag.invoke({"messages": [HumanMessage(content="How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? search the web one time only using the websearch_tool, use that url to visit the website with the tool visit_website_tool. Use the information from the website to answer the question." )]})

# Basic Agent Definition

In [ ]:

class BasicAgent:
    """LangGraph agent"""

    def __init__(self):
        print("BasicAgent initialized.")
        self.graph = build_graph()
        self.system_prompt = (
            """You are a general AI assistant. I will ask you a question and you can use tools when relevant. Reply with the answer and nothing else. YOUR ANSWER should be a number OR as few words as possible OR a comma separated list of numbers and/or strings. If you are asked for a number, don't use comma to write your number, neither use units such as $ or percent sign unless specified otherwise. If you are asked for a string, don't use articles, neither abbreviations (e.g. for cities), and write the digits in plain text unless specified otherwise. If you are asked for a comma separated list, apply the above rules depending of whether the element to be put in the list is a number or a string. If unable to analyze a video or image, analyze any textual clues instead.
            Web browsing rules:
            - Use the minimum number of tool calls necessary. Do not browse further just because additional research is possible.
            - If a tool call provides enough reliable information to answer the question, stop browsing and answer immediately.
            - For questions requiring web research, use search_and_read_web_tool once if no direct URL is provided. It searches the web and reads the resulting pages in one call.
            - If the user provides a direct URL, use visit_website_tool once to retrieve its contents.
            - Do not call websearch_tool separately when search_and_read_web_tool already performs the search.
            - Do not visit a URL again if its contents have already been retrieved and the relevant information is available in the conversation.
            - Only make another browsing call if the previous result failed, did not contain the required information, or leaves a specific essential question unanswered.
            - Before making another call, identify what information is missing and why the existing results are insufficient.
            - Do not search for confirmation of facts that are already adequately supported by the retrieved content.
            - Once sufficient evidence is available, formulate the final answer directly. Do not narrate your browsing process.""")

    def __call__(self, question: str) -> str:
        print(f"Agent received question (first 50 chars): {question}...")

        messages = [
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=question),
        ]

        response = self.graph.invoke({"messages": messages})
        answer = response["messages"][-1].content

        print(f"Agent returning answer: {answer}")
        return answer


def run_and_print_all(profile: gr.OAuthProfile | None):
    """Fetch questions, run the BasicAgent on them, and print the answers."""
    if profile:
        print(f"User logged in: {profile.username}")
    else:
        print("User not logged in.")
        return

    questions_url = f"{DEFAULT_API_URL}/questions"

    try:
        agent = BasicAgent()
    except Exception as e:
        print(f"Error initializing agent: {e}")
        return

    print(f"Fetching questions from: {questions_url}")
    try:
        response = requests.get(questions_url, timeout=15)
        response.raise_for_status()
        questions_data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching questions: {e}")
        return

    if not questions_data:
        print("Fetched questions list is empty.")
        return

    print(f"Running agent on {len(questions_data)} questions...")

    for item in questions_data:
        task_id = item.get("task_id")
        question = item.get("question")

        if not task_id or question is None:
            print(f"Skipping item with missing task_id or question: {item}")
            continue

        try:
            print(f"\nTask ID: {task_id}")
            print(f"Question: {question}")

            final_answer = ""

            for update in agent.graph.stream(
                {"messages": [HumanMessage(content=question)]},
                config={"recursion_limit": 30},
                stream_mode="updates",
            ):
                for node_name, node_update in update.items():
                    print(f"\n[{node_name}]")

                    for message in node_update.get("messages", []):
                        if isinstance(message, AIMessage):
                            if message.tool_calls:
                                print("Tool calls:", message.tool_calls)
                            if message.content:
                                print("Assistant:", message.content)
                                if not message.tool_calls:
                                    final_answer = message.content

                        elif isinstance(message, ToolMessage):
                            print(f"Tool result ({message.name}):", message.content)

            print("\nAnswer:", final_answer)

        except Exception as e:
            print(f"\nError running agent on task {task_id}: {e}")
            traceback.print_exc()

        """
        try:
            answer = agent(question)
            print(f"\nTask ID: {task_id}")
            print(f"Question: {question}")
            print(f"Answer: {answer}")
        except Exception as e:
            print(f"\nError running agent on task {task_id}: {e}")
            traceback.print_exc()"""

# Interface Agent

In [ ]:
# --- Build Gradio Interface using Blocks --- # FOR NOW ONLY PRINT ANSWERS
with gr.Blocks() as demo:
    gr.Markdown("# Basic Agent Evaluation Runner")
    gr.Markdown(
        """
        **Instructions:**

        1.  Please clone this space, then modify the code to define your agent's logic, the tools, the necessary packages, etc ...
        2.  Log in to your Hugging Face account using the button below. This uses your HF username for submission.
        3.  Click 'Run Evaluation & Submit All Answers' to fetch questions, run your agent, submit answers, and see the score.

        ---
        **Disclaimers:**
        Once clicking on the "submit button, it can take quite some time ( this is the time for the agent to go through all the questions).
        This space provides a basic setup and is intentionally sub-optimal to encourage you to develop your own, more robust solution. For instance for the delay process of the submit button, a solution could be to cache the answers and submit in a seperate action or even to answer the questions in async.
        """
    )

    gr.LoginButton()

    run_button = gr.Button("Run Evaluation & Print All Answers")

    status_output = gr.Textbox(label="Run Status / Submission Result", lines=5, interactive=False)
    # Removed max_rows=10 from DataFrame constructor
    results_table = gr.DataFrame(label="Questions and Agent Answers", wrap=True)

    run_button.click(
        fn=run_and_print_all,
        outputs=[status_output, results_table]
    )

if __name__ == "__main__":
    print("\n" + "-"*30 + " App Starting " + "-"*30)
    # Check for SPACE_HOST and SPACE_ID at startup for information
    space_host_startup = os.getenv("SPACE_HOST")
    space_id_startup = os.getenv("SPACE_ID") # Get SPACE_ID at startup

    if space_host_startup:
        print(f"✅ SPACE_HOST found: {space_host_startup}")
        print(f"   Runtime URL should be: https://{space_host_startup}.hf.space")
    else:
        print("ℹ️  SPACE_HOST environment variable not found (running locally?).")

    if space_id_startup: # Print repo URLs if SPACE_ID is found
        print(f"✅ SPACE_ID found: {space_id_startup}")
        print(f"   Repo URL: https://huggingface.co/spaces/{space_id_startup}")
        print(f"   Repo Tree URL: https://huggingface.co/spaces/{space_id_startup}/tree/main")
    else:
        print("ℹ️  SPACE_ID environment variable not found (running locally?). Repo URL cannot be determined.")

    print("-"*(60 + len(" App Starting ")) + "\n")

    print("Launching Gradio Interface for Basic Agent Evaluation...")
    demo.launch(debug=True, share=False)

In [ ]:
import requests

response = requests.get(
    "https://en.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "list": "search",
        "srsearch": "Mercedes Sosa studio albums",
        "format": "json",
    },
    headers={"User-Agent": "MyCourseAgent/1.0 (student project)"},
    timeout=15,
)

print(response.status_code)
print(response.headers.get("Content-Type"))
print(response.text[:300])